# Analyse reproductible des sorties du modèle FVR — Ferlo, Sénégal

Ce notebook analyse toutes les sorties CSV disponibles dans `outputs/` et recherche les couches SIG dans `data/`. Les effectifs sont des superindividus : les hôtes sont convertis en individus réels avec un facteur 20 et les vecteurs avec un facteur 2000 pour les ratios biologiques. Les graphiques sont enregistrés dans `figures/` à 150 dpi minimum.

L'interprétation distingue les transmissions verticales et horizontales lorsque la colonne `origine_infection_vecteur` le permet. Les cartes polygonales sont produites si une couche adéquate est disponible ; sinon, les coordonnées exportées par le modèle sont utilisées comme points et l'absence de géométrie est signalée.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import geopandas as gpd
    import folium
    from shapely.geometry import Point
    GEO_OK = True
except ImportError:
    GEO_OK = False
    print('GeoPandas/Folium indisponibles : les analyses tabulaires continueront.')

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists() and (ROOT.parent / 'outputs').exists():
    ROOT = ROOT.parent
OUTPUTS = ROOT / 'outputs'
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
HOST_SCALE = 20
VECTOR_SCALE = 2000
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'axes.titlesize': 12})
print(f'Racine: {ROOT.resolve()}')
print(f'CSV disponibles: {len(list(OUTPUTS.glob("*.csv")))} | Figures: {FIGURES.resolve()}')

: 

## 1. Chargement et exploration des données
Tous les CSV sont chargés automatiquement. Les fonctions ci-dessous évitent les erreurs lorsque certaines colonnes ne sont pas présentes dans une expérience particulière.

In [ ]:
csv_files = sorted(OUTPUTS.glob('*.csv'))
tables = {}
for path in csv_files:
    try:
        tables[path.stem] = pd.read_csv(path)
    except Exception as exc:
        print(f'Lecture impossible pour {path.name}: {exc}')

def describe_table(name, df):
    print(f'\n### {name}.csv | {df.shape[0]:,} lignes x {df.shape[1]} colonnes')
    display(df.head())
    display(pd.DataFrame({'colonne': df.columns, 'type': [str(t) for t in df.dtypes]}))
    numeric = df.select_dtypes(include='number')
    if not numeric.empty:
        display(numeric.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

for name, df in tables.items():
    describe_table(name, df)

key_groups = {
    'identifiants/temps': ['simulation_id', 'experience', 'cycle', 'jour', 'jour_annee'],
    'SEIR': ['S', 'E', 'I', 'R', 'humains_S', 'animaux_I'],
    'vecteurs': ['aedes', 'culex', 'vecteurs_total', 'oeufs', 'larves'],
    'environnement': ['pluie', 'temperature', 'humidite', 'vent', 'volume', 'ndvi', 'ndwi'],
    'R0/transmission': ['R0', 'transmission', 'origine_infection_vecteur', 'id_zone', 'id_mare']
}
key_inventory = []
for group, tokens in key_groups.items():
    for name, df in tables.items():
        found = [c for c in df.columns if any(token.lower() in c.lower() for token in tokens)]
        if found:
            key_inventory.append({'groupe': group, 'fichier': name, 'colonnes': ', '.join(found)})
display(pd.DataFrame(key_inventory))

## 2. Préparation, unités et fonctions de tracé
Les variables sont agrégées par jour lorsque plusieurs lignes correspondent au même jour. Les courbes restent en agents sauf mention explicite ; les conversions réelles sont réservées aux indicateurs biologiques et aux ratios.

In [ ]:
def get_table(name):
    return tables.get(name, pd.DataFrame()).copy()

def first_col(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None

def time_col(df):
    return first_col(df, ['jour_annee', 'jour', 'cycle'])

def numeric_cols(df, exclude=None):
    exclude = set(exclude or [])
    return [c for c in df.select_dtypes(include='number').columns if c not in exclude]

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Figure enregistrée: {path}')
    return path

def plot_time_series(name, columns, title, ylabel='Valeur (agents ou unité exportée)'):
    df = get_table(name)
    t = time_col(df)
    columns = [c for c in columns if c in df.columns]
    if df.empty or t is None or not columns:
        print(f'{name}: série non disponible avec les colonnes demandées.')
        return None
    plot_df = df.groupby(t, as_index=False)[columns].mean().sort_values(t)
    ax = plot_df.plot(x=t, y=columns, figsize=(11, 5), linewidth=1.8)
    ax.set(title=title, xlabel=t, ylabel=ylabel)
    return savefig(f'{name}_series.png')

def add_window_labels(df, column):
    out = df.copy()
    if out.empty or column not in out.columns:
        return out
    values = np.sort(out[column].dropna().unique())
    if len(values) < 3:
        out['fenetre_temporelle'] = 'unique'
        return out
    q1, q2 = np.quantile(values, [1/3, 2/3])
    out['fenetre_temporelle'] = pd.cut(out[column], [-np.inf, q1, q2, np.inf], labels=['début', 'pic', 'fin'])
    return out

## 3. Analyse temporelle
Les pics sont calculés par maximum de chaque série disponible. Les corrélations sont descriptives et ne doivent pas être interprétées comme causales, en particulier avec une seule simulation.

In [ ]:
series_spec = {
    'populations': ['humains_S', 'humains_E', 'humains_I', 'humains_R', 'animaux_S', 'animaux_E', 'animaux_I', 'animaux_R'],
    'moustiques': ['aedes_S', 'aedes_E', 'aedes_I', 'culex_S', 'culex_E', 'culex_I'],
    'journalier': ['temperature_C', 'humidite_pct', 'pluie_mm', 'vent_ms', 'mares_en_eau'],
    'zones_environnement': ['pluie_mm', 'temperature_C', 'humidite_pct', 'volume_eau_m3', 'ndvi_zone', 'ndwi_zone'],
    'zones_epidemio': ['nouvelles_infections', 'cas_cumules', 'animaux_I', 'humains_I', 'aedes_I', 'culex_I']
}
for name, cols in series_spec.items():
    plot_time_series(name, cols, f'Séries temporelles — {name}')

peak_rows = []
for name, df in tables.items():
    t = time_col(df)
    if t is None:
        continue
    for col in numeric_cols(df, exclude=['simulation_id', 'cycle']):
        valid = df[[t, col]].dropna()
        if not valid.empty:
            row = valid.loc[valid[col].idxmax()]
            peak_rows.append({'fichier': name, 'variable': col, 'jour_pic': row[t], 'valeur_pic': row[col]})
peaks = pd.DataFrame(peak_rows)
display(peaks.sort_values('valeur_pic', ascending=False).head(30))

pop = get_table('populations')
if not pop.empty:
    pop['hotes_infectes_agents'] = pop.get('humains_I', 0) + pop.get('animaux_I', 0)
    pop['hotes_total_reels'] = (pop.get('humains_S', 0) + pop.get('humains_E', 0) + pop.get('humains_I', 0) + pop.get('humains_R', 0) + pop.get('animaux_S', 0) + pop.get('animaux_E', 0) + pop.get('animaux_I', 0) + pop.get('animaux_R', 0)) * HOST_SCALE
    pop['prevalence_hotes'] = pop['hotes_infectes_agents'] / pop['hotes_total_reels'].replace(0, np.nan)
    t = time_col(pop)
    if t:
        pop = add_window_labels(pop, t)
        display(pop.groupby('fenetre_temporelle', observed=True)[['hotes_infectes_agents', 'prevalence_hotes']].agg(['mean', 'max', 'min']))

In [ ]:
daily = get_table('journalier')
mares = get_table('mares')
mosq = get_table('moustiques')
corr_parts = []
if not daily.empty and not mares.empty and 'cycle' in daily and 'cycle' in mares:
    merged = daily.merge(mares, on=['simulation_id', 'cycle'], how='inner', suffixes=('_journalier', '_mares'))
    for a, b in [('pluie_mm', 'volume_moyen'), ('pluie_mm', 'nb_mares_actives')]:
        if a in merged and b in merged:
            corr_parts.append({'relation': f'{a} vs {b}', 'n': merged[[a, b]].dropna().shape[0], 'correlation_Pearson': merged[[a, b]].corr().iloc[0, 1]})
if not daily.empty and not mosq.empty and 'cycle' in daily and 'cycle' in mosq:
    merged = daily.merge(mosq, on=['simulation_id', 'cycle'], how='inner')
    if 'pluie_mm' in merged and 'total' in merged:
        corr_parts.append({'relation': 'pluie_mm vs moustiques_total', 'n': merged[['pluie_mm', 'total']].dropna().shape[0], 'correlation_Pearson': merged[['pluie_mm', 'total']].corr().iloc[0, 1]})
corr_table = pd.DataFrame(corr_parts)
display(corr_table)
if not corr_table.empty:
    corr_table.to_csv(FIGURES / 'correlations_descriptives.csv', index=False)

## 4. Transmission, incidence et prévalence
`transmissions.csv` peut contenir une ligne par agent touché : les événements sont donc comptés avec prudence, en conservant à la fois le nombre de lignes et les agrégations par jour/zone. La colonne `origine_infection_vecteur` est utilisée pour séparer `verticale` et `horizontale`.

In [ ]:
trans = get_table('transmissions')
if trans.empty:
    print('transmissions.csv absent ou vide.')
else:
    origin_col = first_col(trans, ['origine_infection_vecteur', 'origine', 'type_transmission'])
    group_cols = [c for c in ['jour_annee', 'cycle', 'espece_hote', 'espece_vecteur', 'id_zone', 'type_zone'] if c in trans.columns]
    transmission_summary = trans.groupby(group_cols, dropna=False).size().reset_index(name='lignes_evenement') if group_cols else pd.DataFrame()
    display(transmission_summary.head(20))
    if origin_col:
        display(trans[origin_col].value_counts(dropna=False).rename_axis(origin_col).reset_index(name='n_lignes'))
        by_origin = trans.groupby([origin_col, first_col(trans, ['jour_annee', 'cycle'])]).size().reset_index(name='n')
        sns.relplot(data=by_origin, x=by_origin.columns[1], y='n', hue=origin_col, kind='line', marker='o', height=4, aspect=2)
        plt.title('Transmission par origine')
        savefig('transmissions_verticale_horizontale.png')
    if {'jour_annee', 'espece_vecteur'}.issubset(trans.columns):
        epi = trans.groupby(['jour_annee', 'espece_vecteur']).size().reset_index(name='transmissions')
        sns.lineplot(data=epi, x='jour_annee', y='transmissions', hue='espece_vecteur', marker='o')
        plt.title('Courbe des transmissions observées')
        savefig('courbe_transmissions.png')

In [ ]:
inc = get_table('incidence')
if not inc.empty:
    t = time_col(inc)
    if t:
        cols = [c for c in ['incidence_hotes', 'incidence_vecteurs', 'prevalence_hotes', 'prevalence_vecteurs'] if c in inc.columns]
        if cols:
            fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
            [axes[0].plot(inc[t], inc[c], label=c) for c in cols if 'incidence' in c]
            [axes[1].plot(inc[t], inc[c], label=c) for c in cols if 'prevalence' in c]
            axes[0].set_ylabel('Incidence (agents)'); axes[1].set_ylabel('Prévalence')
            axes[0].legend(); axes[1].legend(); axes[1].set_xlabel(t)
            fig.suptitle('Courbes épidémiques')
            savefig('courbes_epidemiques.png')

if not pop.empty and 'hotes_infectes_agents' in pop.columns:
    t = time_col(pop)
    if t:
        ax = pop.plot(x=t, y='hotes_infectes_agents', figsize=(11, 4), color='crimson', legend=False)
        ax.set(title='Effectifs hôtes infectés', ylabel='Agents hôtes', xlabel=t)
        savefig('hotes_infectes.png')

## 5. R0 global, R0 local et gîtes à risque
Le R0 global est rapporté selon les colonnes exportées (`R0_*`). Pour les gîtes, le seuil `R0_local > 1` est appliqué à chaque fenêtre ; une mare est dite à haut risque si elle dépasse ce seuil au moins une fois.

In [ ]:
r0g = get_table('r0_vectoriel')
r0l = get_table('r0_local')
if not r0g.empty:
    r0_cols = [c for c in r0g.columns if c.lower().startswith('r0')]
    print('Résumé R0 global/vectoriel:')
    display(r0g[r0_cols].describe().T if r0_cols else r0g.head())
    if 'jour_debut' in r0g.columns:
        for col in r0_cols:
            if pd.api.types.is_numeric_dtype(r0g[col]):
                sns.lineplot(data=r0g, x='jour_debut', y=col, hue='grandeur' if 'grandeur' in r0g else None, marker='o', legend=False)
                plt.axhline(1, color='crimson', linestyle='--')
                plt.title(f'{col} par fenêtre')
                savefig(f'{col}_fenetres.png')
                plt.show()
if not r0l.empty and 'R0_local' in r0l.columns:
    r0l['haut_risque'] = r0l['R0_local'] > 1
    risk_summary = r0l.groupby('mare', dropna=False).agg(R0_local_max=('R0_local', 'max'), R0_local_moyen=('R0_local', 'mean'), n_fenetres=('R0_local', 'size'), haut_risque=('haut_risque', 'any')).reset_index().sort_values('R0_local_max', ascending=False)
    display(risk_summary.head(20))
    risk_summary.to_csv(FIGURES / 'gites_haut_risque.csv', index=False)
    if {'hotes_moyens', 'volume_eau_m3'}.issubset(r0l.columns):
        sns.pairplot(r0l[['R0_local', 'hotes_moyens', 'volume_eau_m3']].dropna())
        plt.savefig(FIGURES / 'R0_relations_gites.png', dpi=150, bbox_inches='tight')
    elif {'hotes_moyens', 'surface_eau_m2'}.issubset(r0l.columns):
        sns.scatterplot(data=r0l, x='hotes_moyens', y='R0_local', hue='surface_eau_m2', palette='viridis')
        savefig('R0_hotes_surface.png')

## 6. Analyse spatiale et distances
Les exports contiennent des coordonnées métriques. En l'absence de `mares.shp`/`campements.shp`, elles permettent une cartographie ponctuelle de secours. La couche `zone3_entrainements.shp` est recherchée automatiquement et utilisée seulement si sa structure le permet.

In [ ]:
shp_files = sorted(DATA.rglob('*.shp'))
print('Couches détectées:', [p.name for p in shp_files])
r0_points = pd.DataFrame()
if not r0l.empty and {'mare', 'x', 'y', 'R0_local'}.issubset(r0l.columns):
    r0_points = r0l.groupby('mare', as_index=False).agg(x=('x', 'first'), y=('y', 'first'), R0_local=('R0_local', 'max'))
    if GEO_OK:
        r0_geo = gpd.GeoDataFrame(r0_points, geometry=[Point(x, y) for x, y in zip(r0_points.x, r0_points.y)], crs='EPSG:32628')
        ax = r0_geo.plot(column='R0_local', cmap='YlOrRd', legend=True, figsize=(9, 7), markersize=35)
        ax.set_title('R0 local maximal par mare — points exportés')
        savefig('carte_R0_local_points.png')
        folium_map = folium.Map(location=[r0_points.y.mean(), r0_points.x.mean()], zoom_start=10, tiles='CartoDB positron')
        for row in r0_points.itertuples():
            folium.CircleMarker([row.y, row.x], radius=5, color='red' if row.R0_local > 1 else 'steelblue', popup=f'{row.mare}: R0={row.R0_local:.3f}').add_to(folium_map)
        folium_map.save(FIGURES / 'carte_R0_local_points.html')
        display(folium_map)

zone_path = next((p for p in shp_files if 'zone3' in p.stem.lower() or 'suivi' in p.stem.lower()), None)
zones_geo = None
if GEO_OK and zone_path is not None:
    try:
        zones_geo = gpd.read_file(zone_path)
        print(f'Couche chargée: {zone_path.name}, colonnes={list(zones_geo.columns)}')
        display(zones_geo.head())
    except Exception as exc:
        print(f'Lecture SIG impossible: {exc}')

epi_zone = get_table('zones_epidemio')
if not epi_zone.empty and 'id_zone' in epi_zone.columns:
    zone_cases = epi_zone.groupby('id_zone', as_index=False).agg(infections=('nouvelles_infections', 'sum') if 'nouvelles_infections' in epi_zone else ('id_zone', 'size'))
    if {'x', 'y'}.issubset(epi_zone.columns) and GEO_OK:
        coords = epi_zone.groupby('id_zone', as_index=False).agg(x=('x', 'first'), y=('y', 'first')).merge(zone_cases, on='id_zone')
        zone_points = gpd.GeoDataFrame(coords, geometry=[Point(x, y) for x, y in zip(coords.x, coords.y)], crs='EPSG:32628')
        ax = zone_points.plot(column='infections', cmap='Reds', legend=True, figsize=(9, 7), markersize=30)
        ax.set_title('Infections cumulées/sommées par zone épidémiologique')
        savefig('carte_infections_zones.png')

# Distance minimale campement-mare à partir des coordonnées exportées.
if not r0_points.empty and not epi_zone.empty and {'x', 'y', 'type_zone'}.issubset(epi_zone.columns):
    camps = epi_zone[epi_zone['type_zone'].astype(str).str.lower().eq('campement')][['id_zone', 'x', 'y']].drop_duplicates()
    if not camps.empty:
        mare_xy = r0_points[['x', 'y']].to_numpy()
        camps['distance_mare_plus_proche_m'] = [np.sqrt(((mare_xy - np.array([x, y])) ** 2).sum(axis=1)).min() for x, y in zip(camps.x, camps.y)]
        camp_inc = epi_zone.groupby('id_zone', as_index=False)['nouvelles_infections'].sum() if 'nouvelles_infections' in epi_zone else pd.DataFrame()
        distance_df = camps.merge(camp_inc, on='id_zone', how='left') if not camp_inc.empty else camps
        display(distance_df.describe(include='all'))
        if 'nouvelles_infections' in distance_df:
            sns.regplot(data=distance_df, x='distance_mare_plus_proche_m', y='nouvelles_infections', scatter_kws={'alpha': .65})
            plt.title('Distance au gîte le plus proche et incidence de zone')
            savefig('distance_mare_incidence.png')

## 7. Rapport automatique : Résultats et Discussion
Le texte ci-dessous est un canevas fondé sur les indicateurs calculés. Il doit être relu avant diffusion scientifique : une seule trajectoire ne permet ni intervalle d'incertitude ni inférence causale.

In [ ]:
def scalar_max(df, col):
    return float(df[col].max()) if col in df and not df[col].dropna().empty else np.nan

r0_global_max = scalar_max(r0g, 'R0_vectoriel')
r0_local_max = scalar_max(r0l, 'R0_local')
n_high = int(r0l['haut_risque'].sum()) if 'haut_risque' in r0l else 0
total_infections = scalar_max(get_table('resume'), 'infections_totales')
vertical_share = np.nan
if not trans.empty and 'origine_infection_vecteur' in trans.columns:
    vertical_share = (trans['origine_infection_vecteur'].astype(str).str.lower() == 'verticale').mean()

report = f'''# Rapport synthétique — sorties FVR Ferlo

## Résultats

La simulation analysée est une trajectoire unique (simulation_id=1). Les séries décrivent une période journalière allant des jours exportés dans les fichiers CSV. Les effectifs sont des agents du modèle ; pour les interprétations quantitatives, un agent hôte représente {HOST_SCALE} individus et un agent vecteur {VECTOR_SCALE} individus.

### Dynamique environnementale et climatique

Les sorties `journalier.csv`, `climat.csv`, `mares.csv` et `zones_environnement.csv` permettent de suivre la pluie, la température, l'humidité, le vent, le NDVI/NDWI, le volume des mares et leur mise en eau. Les corrélations calculées dans ce notebook sont des associations temporelles : elles peuvent refléter un décalage biologique ou une saison commune, sans démontrer un effet causal.

### Vecteurs et hôtes

Les compartiments Aedes/Culex sont suivis dans `moustiques.csv`, tandis que les compartiments SEIR des hôtes sont suivis dans `populations.csv`. Les pics sont listés dans la table `peaks`; les fenêtres début/pic/fin sont des terciles temporels, utiles pour une synthèse descriptive mais non équivalents à des phases épidémiologiques formellement estimées.

### Transmission et R0

La part de lignes de transmission classées verticales est {vertical_share:.3f} lorsque l'origine est renseignée. Le maximum exporté de R0 vectoriel est {r0_global_max:.3f} et le maximum de R0 local est {r0_local_max:.3f}. {n_high} lignes de fenêtres locales dépassent le seuil R0_local > 1. Le nombre total d'infections rapporté est {total_infections:.0f}. Ces valeurs sont des sorties du modèle et dépendent fortement des paramètres de survie, compétence vectorielle, densité d'hôtes et transmission verticale.

### Analyse spatiale

Les coordonnées des mares et zones servent de représentation ponctuelle lorsque les shapefiles dédiés ne sont pas disponibles. La distance campement-mare la plus proche est calculée dans le système de coordonnées métrique exporté par le modèle. Une couche polygonale devrait être préférée pour une véritable choroplèthe ; le notebook produit une carte GeoPandas/Folium conditionnelle si elle est lisible.

## Discussion

Le profil attendu de la FVR dans le Ferlo combine des pluies intenses, la remise en eau des dépressions temporaires et une réponse différée des vecteurs. Les Aedes associés aux mares temporaires peuvent contribuer au démarrage précoce ; les Culex peuvent soutenir la transmission après augmentation des gîtes et des hôtes disponibles. La transmission verticale est biologiquement importante dans les scénarios de réémergence, mais son taux doit être interprété avec prudence : sa quantification chez certaines espèces locales reste incertaine.

Les gîtes dont R0_local dépasse 1 sont des candidats pour une surveillance ciblée, notamment lorsque le volume, la surface en eau ou la proximité des hôtes augmentent simultanément. Des stratégies possibles sont l'alerte pluviométrique, la surveillance entomologique autour des mares, la réduction de l'exposition du bétail et la vaccination/communication selon les politiques sanitaires. Les analyses de distance ne doivent pas être utilisées seules pour prioriser une intervention sans tenir compte des déplacements pastoraux et de la structure des zones.

### Limites et perspectives

Une seule simulation ne permet pas d'estimer la variabilité entre graines, de tester la robustesse des pics, ni de calculer des intervalles de confiance. Les lignes de `transmissions.csv` peuvent représenter des agents plutôt que des événements indépendants. Les prochaines étapes sont : réplications multi-graines, analyse de sensibilité de `rho_aedes`, survie et EIP, comparaison des expériences Aedes/Animal, validation contre séries entomologiques et pluviométriques observées, et scénarios de contrôle.

Références générales à consulter : Ba et al. (2005) sur l'écologie d'Aedes vexans et Culex poicilipes au Sénégal ; Diallo et al. (2016) sur la compétence vectorielle au Sénégal ; Diallo et al. (2019) sur les repas sanguins dans le Ferlo ; Talla et al. (2016) sur les habitats des vecteurs ; OMS/WOAH pour les aspects cliniques et la surveillance de la FVR.
'''

print(report)
(ROOT / 'rapport_analyse_fvr.md').write_text(report, encoding='utf-8')
print(f'Rapport enregistré: {ROOT / "rapport_analyse_fvr.md"}')

## 8. Reproductibilité et contrôle final
Exécuter les cellules dans l'ordre. Le notebook ne modifie pas les CSV d'origine ; il crée uniquement `figures/`, `rapport_analyse_fvr.md` et les tableaux de synthèse exportés. Vérifier les unités avant toute comparaison externe : les agents du modèle ne sont pas des individus observés directement.